# Week 7 — Hands-On Solutions

Worked answers for the Block 3 hands-on notebook, using the small inline corpus.

Block 3 is yours to do on your own after class, and this notebook is the support
that goes with it. Attempt each task first, then read the solution for the parts
you got stuck on. The comments explain why a choice was made, which is the part
worth reading; the code itself is the easy half.

Task 4 ends with a short list you can check your own answer against.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

docs = [
    "the model trains on data and the data shapes the model",
    "visualization turns data into pictures people can read",
    "a topic model finds themes hiding in a corpus of text",
    "embeddings place similar text near similar text in space",
    "people read pictures faster than they read tables of data",
    "text needs a transform before any visualization can begin",
    "a corpus of documents becomes a matrix of token counts",
    "the matrix view of text loses word order but gains structure",
]

In [ ]:
# Task 1 — with vs without stop words
for sw in (None, "english"):
    cv = CountVectorizer(stop_words=sw)
    c = cv.fit_transform(docs)
    f = pd.Series(c.sum(axis=0).A1, index=cv.get_feature_names_out())
    print(f"stop_words={sw}: top ->", list(f.sort_values().tail(3).index))
# NOTE: with stop words kept, 'the' dominates everything — the chart becomes about grammar.

In [ ]:
# Task 2
tfv = TfidfVectorizer(stop_words="english")
T = tfv.fit_transform(docs)
for d in (1, 3):
    row = pd.Series(T[d].toarray()[0], index=tfv.get_feature_names_out())
    print(f"doc {d+1} distinctive:", list(row.sort_values().tail(5).index))

In [ ]:
# Task 3
from sklearn.decomposition import PCA
pcs = PCA(2).fit_transform(T.toarray())
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(pcs[:, 0], pcs[:, 1], s=120, color="#2E6E8E")
for i, (x_, y_) in enumerate(pcs):
    ax.annotate(f"doc {i+1}", (x_, y_), xytext=(8, 5), textcoords="offset points")
ax.set_title("PCA of TF-IDF vectors (stop words removed) — 8 docs")  # the honest caption
plt.show()

In [ ]:
# Task 4 — cluster the projection three ways
from sklearn.cluster import KMeans, DBSCAN
try:
    from sklearn.cluster import HDBSCAN
except ImportError:
    from hdbscan import HDBSCAN

# Task 3 called its projection `pcs`; alias it so the code below reads like the
# task sheet, which calls it `Y`.
Y = pcs

# `Y` is the 2-D projection from Task 3. The inline corpus is tiny, so the
# parameters below are scaled down; on a student corpus of 100+ documents use
# min_samples=5..10 and min_cluster_size=5..15.
labels = {
    "KMeans k=3":      KMeans(3, n_init=10, random_state=0).fit_predict(Y),
    "DBSCAN eps=0.25": DBSCAN(eps=0.25, min_samples=2).fit_predict(Y),
    "DBSCAN eps=0.50": DBSCAN(eps=0.50, min_samples=2).fit_predict(Y),
    "HDBSCAN":         HDBSCAN(min_cluster_size=2).fit_predict(Y),
}

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, (name, lab) in zip(axes, labels.items()):
    cols = ["#cccccc" if v == -1 else f"C{v % 10}" for v in lab]
    ax.scatter(Y[:, 0], Y[:, 1], c=cols, s=70)
    ax.set_title(f"{name}\n{len(set(lab) - {-1})} groups, "
                 f"{int((np.asarray(lab) == -1).sum())} unassigned", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

# CHECK YOUR ANSWER AGAINST THESE:
# - KMeans always returns exactly k groups and never leaves a point out, so
#   "KMeans found 3 topics" is really a report of your own choice of k.
# - The two DBSCAN rows should differ. If you ran only one eps, you have not yet
#   seen the real lesson: the parameter IS the result.
# - On a corpus this small HDBSCAN often marks most points as -1. That is the
#   honest answer -- there is not enough data to claim structure -- and saying so
#   is a better answer than tuning the parameters until the picture looks tidy.
# - Best answer to "which would you believe": none of them alone; agreement across
#   methods, or a reading of the documents in a cluster, is the evidence.